# Regional two-reference drift and prediction skill

This plotting-only notebook reads the compact multi-region products written by `0_run_drift_diag.ipynb`. It produces regional drift and regional prediction-skill figures for Niño3.4 and the North Atlantic, while retaining the global-land H2OSOI diagnostic. Global maps are handled by `5a_refactor_drift_map.ipynb`; regime-frequency, spatial-summary, and drift–skill relationship figures are intentionally outside this notebook.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd
import xarray as xr

## 1. Configuration

In [ ]:
OUTPUT_ROOT = Path(
    '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/'
    'two_reference'
)
FIGURE_ROOT = Path('/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag')
VARIABLES = ('TREFHT', 'SST', 'PSL', 'PRECT', 'H2OSOI')
INIT_MONTHS = (5, 11)
SOURCES = ('JRA55_FOSIRL', 'Reanalysis')
SOURCE_LABELS = {'JRA55_FOSIRL': 'FOSIRL', 'Reanalysis': 'Reanalysis'}
SOURCE_COLORS = {'JRA55_FOSIRL': 'tab:blue', 'Reanalysis': 'tab:orange'}
MONTH_NAMES = {5: 'May', 11: 'November'}
PLOT_REGIONS = {
    'TREFHT': ('Nino3.4', 'North_Atlantic'),
    'SST': ('Nino3.4', 'North_Atlantic'),
    'PSL': ('Nino3.4', 'North_Atlantic'),
    'PRECT': ('Nino3.4', 'North_Atlantic'),
    'H2OSOI': ('Global_land',),
}
REGION_LABELS = {
    'Nino3.4': 'Niño3.4 (5°S–5°N, 190°–240°E)',
    'North_Atlantic': 'North Atlantic (0°–60°N, 80°W–0°)',
    'Global_land': 'Global land',
}
REGION_FILE_LABELS = {
    'Nino3.4': 'Nino3_4',
    'North_Atlantic': 'North_Atlantic',
    'Global_land': 'Global_land',
}
DRIFT_METRICS = (
    ('e_obs', 'Signed departure from observations'),
    ('e_att', 'Signed departure from model attractor'),
    ('delta_abs_e_obs', 'Change in distance to observations'),
    ('delta_abs_e_att', 'Change in distance to model attractor'),
)
SKILL_METRICS = (
    ('rmse', 'RMSE'),
    ('acc', 'ACC'),
    ('ensemble_spread', 'Ensemble spread'),
    ('spread_rmse_ratio', 'Spread / RMSE'),
)
FIGURE_DPI = 160
FIGURE_SCALE = 1.0
FIGURE_SIZE = (18, 8)
SHOW_FIGURES_INLINE = True
FIGURE_PREFIX = 'fig_leadtime_drift_two_reference'
FONT_SIZE = 10
TITLE_FONT_SIZE = 12
LEGEND_FONT_SIZE = 8

FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    'font.size': FONT_SIZE,
    'axes.titlesize': FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'figure.titlesize': TITLE_FONT_SIZE,
    'legend.fontsize': LEGEND_FONT_SIZE,
})


def figure_size():
    return tuple(FIGURE_SCALE * value for value in FIGURE_SIZE)

## 2. Validate and load regional products

Only source-specific `regional` products are required. Unrelated spatial, paired-regime, and other diagnostic products are neither opened nor required for validation.

In [ ]:
manifest_path = OUTPUT_ROOT / 'two_reference_drift_figure_data_manifest.csv'
if not manifest_path.is_file():
    raise FileNotFoundError(
        f'Missing {manifest_path}; run jupyter/0_run_drift_diag.ipynb first'
    )
full_manifest = pd.read_csv(manifest_path)
regional_manifest = full_manifest.loc[
    full_manifest['variable'].isin(VARIABLES)
    & full_manifest['init_month'].isin(INIT_MONTHS)
    & full_manifest['source'].isin(SOURCES)
    & full_manifest['product'].eq('regional')
].copy()
expected_products = len(VARIABLES) * len(INIT_MONTHS) * len(SOURCES)
if len(regional_manifest) != expected_products:
    raise ValueError(
        f'Expected {expected_products} regional products, found '
        f'{len(regional_manifest)}'
    )
if regional_manifest.duplicated(['variable', 'init_month', 'source']).any():
    raise ValueError('Regional manifest contains duplicate variable/month/source rows')
missing_paths = [
    Path(path) for path in regional_manifest['path']
    if not Path(path).is_file() or Path(path).stat().st_size == 0
]
if missing_paths:
    display(pd.DataFrame({'missing_regional_path': missing_paths}))
    raise FileNotFoundError(
        f'{len(missing_paths)} required regional products are missing'
    )
display(regional_manifest.sort_values(['variable', 'init_month', 'source']))


def regional_product_path(variable, init_month, source):
    match = regional_manifest.loc[
        regional_manifest['variable'].eq(variable)
        & regional_manifest['init_month'].eq(init_month)
        & regional_manifest['source'].eq(source),
        'path',
    ]
    if len(match) != 1:
        raise ValueError(
            f'Expected one regional product for {variable}, init={init_month:02d}, '
            f'{source}; found {len(match)}'
        )
    return Path(match.iloc[0])


def load_regional_product(variable, init_month, source):
    path = regional_product_path(variable, init_month, source)
    with xr.open_dataset(path) as opened:
        saved = opened.load()
    required_drift = {name for name, _ in DRIFT_METRICS}
    required_skill = {f'skill_{name}' for name, _ in SKILL_METRICS}
    missing = (required_drift | required_skill) - set(saved.data_vars)
    if missing:
        raise KeyError(f'{path} is missing regional fields: {sorted(missing)}')
    expected_regions = PLOT_REGIONS[variable]
    if 'region' not in saved.dims:
        raise ValueError(
            f'{path} predates multi-region output; rerun '
            'jupyter/0_run_drift_diag.ipynb'
        )
    available_regions = tuple(saved.region.values.astype(str))
    missing_regions = set(expected_regions) - set(available_regions)
    if missing_regions:
        raise ValueError(
            f'{path} is missing regions: {sorted(missing_regions)}; '
            'rerun jupyter/0_run_drift_diag.ipynb'
        )
    regional = saved[sorted(required_drift)]
    skill = saved[sorted(required_skill)].rename(
        {name: name.removeprefix('skill_') for name in required_skill}
    )
    return {'regional': regional, 'skill': skill, 'path': path}


results = {
    (variable, init_month, source): load_regional_product(
        variable, init_month, source
    )
    for variable in VARIABLES
    for init_month in INIT_MONTHS
    for source in SOURCES
}
print(f'Loaded {len(results)} source-specific regional products.')

## 3. Regional drift figures

The first two columns are signed departures. For the two absolute-distance-change columns, negative values indicate movement closer to the reference and positive values indicate movement farther away.

In [ ]:
figure_paths = []
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            len(INIT_MONTHS), len(DRIFT_METRICS),
            figsize=figure_size(), sharex=True, squeeze=False,
            constrained_layout=True,
        )
        for row, init_month in enumerate(INIT_MONTHS):
            for col, (metric, title) in enumerate(DRIFT_METRICS):
                ax = axes[row, col]
                for source in SOURCES:
                    field = results[(
                        variable, init_month, source
                    )]['regional'][metric].sel(region=region_name)
                    field.mean('Y', skipna=True).plot(
                        ax=ax, label=SOURCE_LABELS[source],
                        color=SOURCE_COLORS[source],
                    )
                ax.axhline(0, color='0.35', linewidth=0.8)
                ax.set_title(f'{MONTH_NAMES[init_month]}: {title}')
                ax.set_xlabel('Lead month')
                ax.legend()
        fig.suptitle(
            f'{variable}: regional two-reference drift, '
            f'{REGION_LABELS[region_name]}'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_regional_drift.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        figure_paths.append(path)

## 4. Regional prediction-skill figures

In [ ]:
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            len(INIT_MONTHS), len(SKILL_METRICS),
            figsize=figure_size(), sharex=True, squeeze=False,
            constrained_layout=True,
        )
        for row, init_month in enumerate(INIT_MONTHS):
            for col, (metric, title) in enumerate(SKILL_METRICS):
                ax = axes[row, col]
                for source in SOURCES:
                    field = results[(
                        variable, init_month, source
                    )]['skill'][metric].sel(region=region_name)
                    field.plot(
                        ax=ax, label=SOURCE_LABELS[source],
                        color=SOURCE_COLORS[source],
                    )
                ax.set_title(f'{MONTH_NAMES[init_month]}: {title}')
                ax.set_xlabel('Lead month')
                ax.legend()
        fig.suptitle(
            f'{variable}: regional prediction skill, '
            f'{REGION_LABELS[region_name]}'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_regional_skill.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        figure_paths.append(path)

## 5. Validation

In [ ]:
expected_figure_count = 2 * sum(
    len(PLOT_REGIONS[variable]) for variable in VARIABLES
)
assert len(figure_paths) == expected_figure_count
assert all(path.is_file() and path.stat().st_size > 0 for path in figure_paths)
assert len(results) == len(VARIABLES) * len(INIT_MONTHS) * len(SOURCES)
for (variable, _, _), result in results.items():
    available = set(result['regional'].region.values.astype(str))
    assert set(PLOT_REGIONS[variable]) <= available
print(
    f'Validated {len(figure_paths)} regional drift/skill figures from '
    f'{len(results)} source-specific regional products.'
)